In [7]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [8]:
data = pd.read_csv("chemical_process_timeseries.csv")
data.head(20)

,timestamp,operating_regime,reactor_id,ambient_temp_effect,reactor_temp,reactor_pressure,feed_flow_rate,coolant_flow_rate,agitator_speed_rpm,reaction_rate,...,selectivity,yield_pct,vibration_rms,motor_current,power_consumption_kw,temp_setpoint,pressure_setpoint,fault_type,efficiency_loss_pct,time_to_fault_min
0,2024-01-01 00:00:00,A,A_R1,0.000000,181.135558,15.791013,101.108882,79.154645,305.779931,0.724542,...,91.927424,82.032893,1.470297,45.882315,41.294083,180.0,12.0,0,0.0,NaN
1,2024-01-01 00:01:00,A,A_R1,0.000485,182.249820,15.706975,98.932369,NaN,302.283603,0.728999,...,91.581466,81.301503,1.425847,46.755905,42.080314,180.0,12.0,0,0.0,NaN
2,2024-01-01 00:02:00,A,A_R1,0.000970,183.129737,15.593397,99.792219,80.593736,305.814440,0.732519,...,91.895056,82.006626,1.448190,NaN,NaN,NaN,12.0,0,NaN,NaN
3,2024-01-01 00:03:00,A,A_R1,0.001454,183.020987,15.527333,99.854222,80.190444,304.436123,0.732084,...,91.544885,81.562261,1.473448,45.862171,41.275954,180.0,12.0,0,0.0,NaN
4,2024-01-01 00:04:00,A,A_R1,0.001939,183.077096,NaN,99.302349,78.571707,304.811418,0.732308,...,92.258005,82.411124,1.523491,46.880227,42.192204,180.0,12.0,0,NaN,NaN
5,2024-01-01 00:05:00,A,A_R1,0.002424,181.606292,15.580754,101.186188,78.822545,NaN,0.726425,...,92.203569,82.208935,1.530768,45.546395,40.991755,180.0,12.0,0,0.0,NaN
6,2024-01-01 00:06:00,A,A_R1,0.002909,182.841240,15.836389,100.681247,80.726173,300.562559,0.731365,...,91.747148,81.800334,1.555276,45.587534,41.028780,180.0,12.0,0,0.0,NaN
7,2024-01-01 00:07:00,A,A_R1,0.003394,182.587322,15.448385,101.109490,79.484674,298.963240,0.730349,...,92.089153,82.302182,1.481548,46.855209,42.169688,180.0,12.0,0,0.0,NaN
8,2024-01-01 00:08:00,A,A_R1,0.003879,182.869980,15.488952,NaN,81.214908,291.689501,0.731480,...,91.353668,81.935568,1.553037,45.627244,41.064520,NaN,12.0,0,0.0,NaN
9,2024-01-01 00:09:00,A,A_R1,0.004363,182.987364,15.661034,NaN,79.829456,299.116733,0.731949,...,91.803851,82.007645,1.404403,47.101036,NaN,180.0,12.0,0,0.0,NaN


In [9]:
# data header
data.columns

Index(['timestamp', 'operating_regime', 'reactor_id', 'ambient_temp_effect',
       'reactor_temp', 'reactor_pressure', 'feed_flow_rate',
       'coolant_flow_rate', 'agitator_speed_rpm', 'reaction_rate',
       'conversion_rate', 'selectivity', 'yield_pct', 'vibration_rms',
       'motor_current', 'power_consumption_kw', 'temp_setpoint',
       'pressure_setpoint', 'fault_type', 'efficiency_loss_pct',
       'time_to_fault_min'],
      dtype='str')

In [10]:
data.shape

(777600, 21)

In [11]:
rows = []
for header in data.columns:
    col = data[header]
    if pd.api.types.is_numeric_dtype(col):
        col_type = "number"
        info = f"{col.min()} - {col.max()}"
    else:
        col_type = "text"
        info = f"{col.nunique()} unique"
    rows.append({
        "header": header,
        "type": col_type,
        "range/unique": info,
        "#NaN": col.isnull().sum(),
    })

from tabulate import tabulate
print(tabulate(rows, headers="keys", tablefmt="github"))

| header               | type   | range/unique                            |   #NaN |
|----------------------|--------|-----------------------------------------|--------|
| timestamp            | text   | 129600 unique                           |      0 |
| operating_regime     | text   | 2 unique                                |      0 |
| reactor_id           | text   | 6 unique                                |      0 |
| ambient_temp_effect  | number | -9.999999999265476 - 9.999999999265476  |  46561 |
| reactor_temp         | number | 177.79391332263188 - 202.34919298034345 |  46965 |
| reactor_pressure     | number | 15.214214694490508 - 18.335541962050197 |  47143 |
| feed_flow_rate       | number | 77.47709429761954 - 113.53596915251192  |  46760 |
| coolant_flow_rate    | number | 57.02463065369688 - 86.26379473587124   |  46645 |
| agitator_speed_rpm   | number | 273.1360992927756 - 330.6032179350444   |  46775 |
| reaction_rate        | number | 0.7118138978594639 - 0.79462291

In [12]:
for header in ["operating_regime", "reactor_id", "fault_type", "temp_setpoint", "pressure_setpoint"]:
    print(f"{header}: {data[header].unique()}")

operating_regime: <StringArray>
['A', 'B']
Length: 2, dtype: str
reactor_id: <StringArray>
['A_R1', 'A_R2', 'A_R3', 'B_R1', 'B_R2', 'B_R3']
Length: 6, dtype: str
fault_type: [0 2 1 4 3]
temp_setpoint: [180.  nan 192.]
pressure_setpoint: [12.   nan 13.5]


In [13]:
# Crosstab count sample of temp_setpoint and pressure_setpoint given operating_regime and reactor_id
pd.crosstab(
    index=[data["operating_regime"], data["reactor_id"]],
    columns=[data["temp_setpoint"], data["pressure_setpoint"]]
)

temp_setpoint                 180.0   192.0
pressure_setpoint              12.0    13.5
operating_regime reactor_id                
A                A_R1        114393       0
                 A_R2        114481       0
                 A_R3        114703       0
B                B_R1             0  114387
                 B_R2             0  114413
                 B_R3             0  114617

In [14]:
# Test efficiency_loss_pct if 0 or non-0, given operating_regime, and given NaN or not in "time_to_fault_min"
eff = data["efficiency_loss_pct"]
eff_cat = np.select([eff.isnull(), eff == 0], ["NaN", "zero"], default="non-zero")
ttf_cat = np.where(data["time_to_fault_min"].isnull(), "NaN", "has value")

pd.crosstab(
    data["operating_regime"].fillna("NaN"),
    [eff_cat, ttf_cat],
    rownames=["operating_regime"],
    colnames=["efficiency_loss", "time_to_fault"],
    margins=True,
)


efficiency_loss     NaN           non-zero              zero     All
time_to_fault       NaN has value      NaN has value     NaN        
operating_regime                                                    
A                 22699       757      723     12466  352155  388800
B                 22676       787      791     12277  352269  388800
All               45375      1544     1514     24743  704424  777600

In [15]:
# Test efficiency_loss_pct if 0 or non-0, given fault_type, and given NaN or not in "time_to_fault_min"
# fault_type 0 (no fault): every known efficiency_loss_pct is exactly 0 -> fill NaN with 0
no_fault = data["fault_type"] == 0
data.loc[no_fault, "efficiency_loss_pct"] = data.loc[no_fault, "efficiency_loss_pct"].fillna(0.0)

# Rebuild the categories from the updated data, then check again
eff = data["efficiency_loss_pct"]
eff_cat = np.select([eff.isnull(), eff == 0], ["NaN", "zero"], default="non-zero")
ttf_cat = np.where(data["time_to_fault_min"].isnull(), "NaN", "has value")

pd.crosstab(
    data["fault_type"],
    [eff_cat, ttf_cat],
    rownames=["fault_type"],
    colnames=["efficiency_loss", "time_to_fault"],
    margins=True,
)


efficiency_loss  NaN           non-zero              zero     All
time_to_fault    NaN has value      NaN has value     NaN        
fault_type                                                       
0                  0         0        0         0  749699  749699
1                 35       509      489      8110       0    9143
2                 19       381      337      6105       0    6842
3                 15       262      249      3948       0    4474
4                 31       392      439      6580       0    7442
All              100      1544     1514     24743  749699  777600

Given that there are 777,600 data points and uniform distribution of the dataset, there are 129,600 timestamp, 2 operating regime, and 6 reactors (129,600 * 2 * 3 = 777,600). There are 2 operating regimes in this dataset including:
- A: $180^\circ C$ and 12 **bar**
- B: $192^\circ C$ and 13.5 **bar**

The majority of data shows the process running normal and there are 27,901 data points for 4 fault types in total.

| Factor | Primary Category | Compound Designation | Explanation |
| --- | --- | --- | --- |
| `ambient_temp_effect` | **Input** | Uncontrolled Input (Disturbance) | External environmental condition driving cooling demand. |
| `feed_flow_rate` | **Input** | Manipulated Input | Directly controlled inlet flow setting. |
| `coolant_flow_rate` | **Input** | Manipulated Input | Control valve setting used to remove reaction heat. |
| `agitator_speed_rpm` | **Input** | Manipulated Input | Setpoint controlling mixer rotational speed. |
| `reactor_temp` | **Output** | Direct Output | Directly measured thermal outcome resulting from inputs & reaction heat balance. |
| `reactor_pressure` | **Output** | Direct Output | Directly measured pressure response. |
| `reaction_rate` | **Inferred** | Inferred Output | Calculated kinetic metric derived from temperature, concentration, and volume. |
| `conversion_rate` | **Inferred** | Inferred Output | Soft-sensor KPI calculated from reactant consumption. |
| `selectivity` | **Inferred** | Inferred Output | Quality KPI computed from product stream compositions. |
| `yield_pct` | **Inferred** | Inferred Output | Efficiency KPI calculated as $\frac{\text{Actual Output}}{\text{Theoretical Max Output}} \times 100$. |
| `vibration_rms` | **Tracked** | Tracked Output (Condition) | Downstream mechanical physical response (result of mixing speed and bearing health). |
| `motor_current` | **Tracked** | Tracked Input/Output (Electrical load) | Response to agitator speed and fluid viscosity changes. |
| `power_consumption_kw` | **Tracked** | Tracked Input (Resource) | Utility energy draw measured to monitor power usage. |

1. **Direct Output vs. Inferred Output:**
* **Direct Outputs** (`reactor_temp`, `reactor_pressure`) are physical signals read instantly by temperature transmitters and pressure transducers.
* **Inferred Outputs** (`yield_pct`, `conversion_rate`, `selectivity`, `reaction_rate`) are virtual signals generated by process computers, soft-sensors, or lab analysis using equations.


2. **Manipulated Inputs vs. Tracked Features:**
* **Manipulated Inputs** are knobs you turn to control the reaction.
* **Tracked Features** reflect mechanical wear, electrical load, and power consumption—used primarily for anomaly detection and predictive maintenance rather than process control loops.

### Category Summary

* **Inputs (Manipulated & Disturbances):** `ambient_temp_effect`, `feed_flow_rate`, `coolant_flow_rate`, `agitator_speed_rpm`
* **Outputs (Process Responses/States):** `reactor_temp`, `reactor_pressure`
* **Inferred (Chemical Kinetics & Performance KPIs):** `reaction_rate`, `conversion_rate`, `selectivity`, `yield_pct`
* **Tracked (Equipment Health & Mechanical/Electrical Metrics):** `vibration_rms`, `motor_current`, `power_consumption_kw`

In [16]:
# Given operating_regime is A, replace NaN in pressure_setpoint and temp_setpoint with 12.0 and 180.0
# Given operating_regime is B, replace NaN in pressure_setpoint and temp_setpoint with 13.5 and 192.0

data.loc[(data["operating_regime"] == "A") & (data["pressure_setpoint"].isnull()), "pressure_setpoint"] = 12.0
data.loc[(data["operating_regime"] == "A") & (data["temp_setpoint"].isnull()), "temp_setpoint"] = 180.0
data.loc[(data["operating_regime"] == "B") & (data["pressure_setpoint"].isnull()), "pressure_setpoint"] = 13.5
data.loc[(data["operating_regime"] == "B") & (data["temp_setpoint"].isnull()), "temp_setpoint"] = 192.0

data.isnull().sum()

timestamp                    0
operating_regime             0
reactor_id                   0
ambient_temp_effect      46561
reactor_temp             46965
reactor_pressure         47143
feed_flow_rate           46760
coolant_flow_rate        46645
agitator_speed_rpm       46775
reaction_rate            46830
conversion_rate          46416
selectivity              47209
yield_pct                46524
vibration_rms            46459
motor_current            46617
power_consumption_kw     46494
temp_setpoint                0
pressure_setpoint            0
fault_type                   0
efficiency_loss_pct       1644
time_to_fault_min       751313
dtype: int64

In [19]:
import os

drop_cols = ["operating_regime", "reactor_id", "temp_setpoint", "pressure_setpoint", "timestamp"]

# Reposition timestamp to minutes since start (2024-01-01 00:00:00 -> 0)
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["minute"] = ((data["timestamp"] - data["timestamp"].min()) / pd.Timedelta(minutes=1)).astype(int)

# Reactor datasets export
for reactor, subset in data.groupby("reactor_id"):
    subset = subset.drop(columns=drop_cols).set_index("minute")
    filename = f"chemical_process_timeseries_{reactor}.csv"
    subset.to_csv(filename)  # index (minute) is written as the first column

# Fault type datasets export (no-fault rows excluded)
faults = data[data["fault_type"] != 0].drop(columns="timestamp").set_index(["reactor_id", "minute"])
faults.to_csv("chemical_process_timeseries_faults.csv")


In [20]:
INPUT_COLS = ["ambient_temp_effect", "feed_flow_rate", "coolant_flow_rate", "agitator_speed_rpm"]
OUTPUT_COLS = ["reactor_temp", "reactor_pressure", "reaction_rate", "conversion_rate", "selectivity", "yield_pct"]
TRACKING_COLS = ["vibration_rms", "motor_current", "power_consumption_kw"]

def plot_over_time(filename, cols=INPUT_COLS, fault_type=None, kind="line"):
    df = pd.read_csv(filename)
    df = df[df["fault_type"] == fault_type] if fault_type is not None else df
    fig, axes = plt.subplots(len(cols), 1, figsize=(25, 2.2 * len(cols)), sharex=True, squeeze=False)
    for ax, col in zip(axes[:, 0], cols):
        if kind == "scatter":
            ax.scatter(df["minute"], df[col], s=2, alpha=0.5, linewidths=0, color="#3b6ea5", rasterized=True)
        else:
            ax.plot(df["minute"], df[col], linewidth=0.6, color="#3b6ea5")
        ax.set_ylabel(col, rotation=0, ha="right", va="center")
        ax.grid(axis="y", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1, 0].set_xlabel("Minute")
    fig.tight_layout()
    fig.savefig(os.path.splitext(filename)[0] + f"_fault{fault_type}_{kind}.png", dpi=300)
    plt.close() # Close the figure to free up memory after saving

for id in data["reactor_id"].unique():
    plot_over_time(f"chemical_process_timeseries_{id}.csv", cols=INPUT_COLS + OUTPUT_COLS + TRACKING_COLS, fault_type=0, kind="scatter")
    print(f"Saved scatter plot for reactor {id} with fault_type=0")

Saved scatter plot for reactor A_R1 with fault_type=0
Saved scatter plot for reactor A_R2 with fault_type=0
Saved scatter plot for reactor A_R3 with fault_type=0
Saved scatter plot for reactor B_R1 with fault_type=0
Saved scatter plot for reactor B_R2 with fault_type=0
Saved scatter plot for reactor B_R3 with fault_type=0
